### Statistical Modeling

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
#importing data
customers = pd.read_csv("../data/processed/Customers.csv")
orders = pd.read_csv("../data/processed/Orders.csv")
order_items = pd.read_csv("../data/processed/Order_items.csv")
products = pd.read_csv("../data/processed/Products.csv")
shipping = pd.read_csv("../data/processed/Shipping.csv")
stores = pd.read_csv("../data/processed/Stores.csv")
departments = pd.read_csv("../data/processed/Store_departments.csv")

In [ ]:
# Suming up multiple values

order_items_agg = order_items.groupby("Order_id").agg({
    "Order_item_quantity": "sum",
    "Order_item_total": "sum",
    "Order_item_discount": "mean"
}).reset_index()
order_items_agg

In [5]:
# merging data

df = orders.merge(customers, on="Customer_id", how="left")
df = df.merge(order_items_agg, on="Order_id", how="left")
df = df.merge(shipping, on="Order_id", how="left")
df = df.merge(stores, on="Store_id", how="left")

In [ ]:
df.head()
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65752 entries, 0 to 65751
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Order_id             65752 non-null  int64  
 1   Customer_id          65752 non-null  int64  
 2   Store_id             65752 non-null  int64  
 3   Order_date           65752 non-null  object 
 4   Order_region         65752 non-null  object 
 5   Market               65752 non-null  object 
 6   Order_country        65752 non-null  object 
 7   Order_state          65752 non-null  object 
 8   Order_city           65752 non-null  object 
 9   Order_zipcode        8270 non-null   float64
 10  Delivery_status      65752 non-null  object 
 11  Order_status         65752 non-null  object 
 12  Payment_method       65752 non-null  object 
 13  Order_total          65752 non-null  float64
 14  Customer_fname       65752 non-null  object 
 15  Customer_lname       65744 non-null 

In [26]:
df['Order_month'] = pd.to_datetime(df['Order_date']).dt.month
df['Order_day'] = pd.to_datetime(df['Order_date']).dt.day
df['Order_weekday'] = pd.to_datetime(df['Order_date']).dt.weekday

C:\Users\WELCOME\AppData\Local\Temp\ipykernel_10664\758180110.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Order_month'] = pd.to_datetime(df['Order_date']).dt.month
C:\Users\WELCOME\AppData\Local\Temp\ipykernel_10664\758180110.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Order_day'] = pd.to_datetime(df['Order_date']).dt.day
C:\Users\WELCOME\AppData\Local\Temp\ipykernel_10664\758180110.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Order_weekday'] = pd.to_datetime(df['Order_date']).dt.weekday


In [ ]:
#data cleaning - Removing unnecessary columns that will affect modelling
columns_to_remove = [
    'Order_id',
    'Customer_id',
    'Store_id',
    'Order_date',
    'Customer_fname',
    'Customer_lname',
    'Customer_email',
    'Customer_password',
    'Customer_street',
    'Delivery_status',
    'Order_status',
    'Days_for_shipping',
    'Order_city',
    'Order_state',
    'Order_country',
    'Customer_city',
    'Customer_state',
    'Shipping_date',
    'Customer_country',
    'Order_item_total',
    'Order_zipcode',
    'Customer_zipcode'
]

X = df.drop(columns=columns_to_remove + ['Late_delivery_risk'])
print("After removing IDs. Shape:", X.shape)
print("Features:", X.columns.tolist())


After removing IDs. Shape: (65752, 14)
Features: ['Order_region', 'Market', 'Payment_method', 'Order_total', 'Customer_segment', 'Order_item_quantity', 'Order_item_discount', 'Shipping_mode', 'Days_for_shipment', 'Latitude', 'Longitude', 'Order_month', 'Order_day', 'Order_weekday']


In [32]:
X.nunique().sort_values(ascending=False).head(15)

Order_total            43201
Order_item_discount    26190
Latitude               11250
Longitude               4487
Order_day                 31
Order_item_quantity       24
Order_region              23
Order_month               12
Order_weekday              7
Market                     5
Days_for_shipment          4
Payment_method             4
Shipping_mode              4
Customer_segment           3
dtype: int64

In [ ]:
X = pd.get_dummies(X, drop_first=True) #Encoding categories

#### Target Variable (Y)
##### Late Delivery prediction

In [38]:
y = df['Late_delivery_risk']

##### Train test split 

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [40]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [41]:
X_train = X_train[:10000]
y_train = y_train[:10000]
print("Training samples:", len(X_train)) # for faster and safe

Training samples: 10000


##### Train model 

In [42]:
#Logistic Regression for binary classification 

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

c:\Users\WELCOME\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


##### Prediction

In [43]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

##### Model Evaluation

In [44]:
print(f"Accuracy: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6841

Confusion Matrix:
[[4945 1007]
 [3147 4052]]

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.83      0.70      5952
           1       0.80      0.56      0.66      7199

    accuracy                           0.68     13151
   macro avg       0.71      0.70      0.68     13151
weighted avg       0.72      0.68      0.68     13151



##### The Logistic Regression model achieved an accuracy of 68%, meaning it correctly predicted the delivery status for most orders. The accuracy is moderate because Logistic Regression is a simple model and may not capture all complex patterns in the data.

#### Feature Importance

In [45]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 Features:")
print(coef_df.head(10))


Top 10 Features:
                         Feature  Coefficient
40        Shipping_mode_Same Day    -3.391401
42  Shipping_mode_Standard Class    -1.619956
41    Shipping_mode_Second Class    -1.032722
3              Days_for_shipment    -0.515157
15     Order_region_Eastern Asia     0.461109
32                  Market_LATAM     0.366589
36        Payment_method_PAYMENT     0.357671
31                 Market_Europe     0.345076
35          Payment_method_DEBIT     0.339591
33           Market_Pacific Asia     0.338679


#### Chi-Square Test

In [ ]:
# Does the order region affect late delivery risk?

table = pd.crosstab(df['Order_region'], df['Late_delivery_risk'])
chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

Chi-square: 31.546456653096985
p-value: 0.08546392978087133


Order region does not have a statistically significant effect on late delivery risk.

p ≥ 0.05 → Fail to reject H₀ → No strong evidence of relationship